# Lab 10: Agent-Based Models of Scientific Communities

**AIDE Summer 2025**

---

This notebook builds two simulation models from the philosophy of science to understand
*when and why* individual rationality fails at the collective level.

**Part 1 — The Zollman Effect**: A network of Bayesian scientists shows that *dense*
communication can prevent a community from finding the right answer — even when every
individual agent is reasoning perfectly.

**Part 2 — Better than Best**: An NK landscape model shows that *mixed* exploration
strategies outperform uniform ones — but only on hard problems.

---

**How to use this notebook**
- The `Settings` cell below has two modes: `FAST` (~3 min total) and `FULL` (~10 min).
  Change `MODE` and re-run from top to get publication-quality results.
- Run cells in order. Start each simulation cell, then read the explanation below while it runs.
- Discussion questions are at the end.

In [ ]:
!pip install numpy matplotlib networkx --quiet

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from itertools import product
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

In [ ]:
# =============================================================================
# SETTINGS
# Change MODE to 'FAST' (~3 min, good for class demos) or
#                'FULL' (~10 min, more robust statistics)
# After changing, re-run all cells from top.
# =============================================================================

MODE = 'FAST'   # <-- change to 'FULL' for richer results

if MODE == 'FAST':
    # ---- Part 1: Zollman / multi-arm bandit --------------------------------
    N_SCIENTISTS = 8       # scientists per network
    N_SIMS       = 80      # simulations per topology
    N_ROUNDS_Z   = 300     # rounds per simulation
    DELTA_SWEEP  = [0.001, 0.005, 0.02, 0.1]   # effect sizes to test
    # ---- Part 2: NK landscape ----------------------------------------------
    N_BITS       = 8       # N: bits per theory (binary string length)
    N_AGENTS     = 10      # scientists per community
    N_ROUNDS_NK  = 60      # rounds per community run
    N_TRIALS     = 10      # landscapes to average over
    K_SWEEP      = [0, 1, 2, 3, 4, 5, 6, 7]
    N_RUGGED     = 15      # landscapes for ruggedness chart

elif MODE == 'FULL':
    # ---- Part 1: Zollman / multi-arm bandit --------------------------------
    N_SCIENTISTS = 10
    N_SIMS       = 300
    N_ROUNDS_Z   = 600
    DELTA_SWEEP  = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
    # ---- Part 2: NK landscape ----------------------------------------------
    N_BITS       = 10
    N_AGENTS     = 15
    N_ROUNDS_NK  = 120
    N_TRIALS     = 25
    K_SWEEP      = [0, 1, 2, 3, 4, 5, 6, 7]
    N_RUGGED     = 25

else:
    raise ValueError('MODE must be FAST or FULL')

print('Mode:', MODE)
print('  Part 1: {} scientists, {} sims, {} rounds each'.format(N_SCIENTISTS, N_SIMS, N_ROUNDS_Z))
print('  Part 2: N={} bits, {} agents, {} rounds, {} landscapes'.format(N_BITS, N_AGENTS, N_ROUNDS_NK, N_TRIALS))

---

## Part 1: The Zollman Effect

### The Opening Puzzle

> In the early 2000s, pharmaceutical companies converged on SSRIs as antidepressants
> because early trials looked promising. Many labs stopped exploring alternatives.
> Years later, ketamine proved transformative for treatment-resistant depression —
> a mechanism the field had nearly abandoned.
>
> **Was it rational for each lab to follow the best-looking approach?** Yes.
> **Did that produce a good collective outcome?** No.

This is the paradox Kevin Zollman (2007) formalized:
**individual rationality can produce collective epistemic failure**.

### Model Setup

We simulate a community of scientists who:
1. Choose a **research method** (A or B) based on their current Bayesian beliefs
2. Run one experiment and observe a noisy outcome (success or failure)
3. **Share their result only with direct network neighbors** — not the whole field
4. Update their beliefs when they receive a neighbor's data

**Hidden truth**: method B is slightly better than A (e.g., p\_B = 0.501, p\_A = 0.500).
The community's task is to eventually converge on B.

**The question**: does the *topology* of the communication network determine
whether the community finds the right answer?

### Bayesian Belief Updating

Each scientist tracks **hits** (successes) and **misses** (failures) for each method.
Their belief about method `i`'s success rate is the mean of a Beta distribution:

```
belief[i] = (1 + hits[i]) / (2 + hits[i] + misses[i])
```

**Why this formula?**
We start with a uniform prior: Beta(1, 1), meaning equal probability for any success
rate. After accumulating evidence, the posterior shifts toward the true rate.
The formula is the mean of that posterior.

**Sharing evidence:**
When scientist A shares results with neighbor B, B simply adds A's hits and misses to
their own running totals. This is the exact correct Bayesian update — the Beta
distribution is the conjugate prior for Bernoulli observations, so combining evidence
is just adding counts.

**Exercise 1**: implement `receive_evidence` to do this update.

In [ ]:
class Scientist:
    '''A Bayesian agent who experiments, shares results, and updates beliefs.'''

    def __init__(self, n_methods=2):
        # Uniform prior Beta(1, 1) for each method
        self.hits   = [1] * n_methods
        self.misses = [1] * n_methods

    @property
    def beliefs(self):
        '''Mean of Beta(1 + hits, 1 + misses) for each method.'''
        return [(1 + h) / (2 + h + m)
                for h, m in zip(self.hits, self.misses)]

    def best_method(self):
        '''Index of the method this scientist currently prefers.'''
        return int(np.argmax(self.beliefs))

    def run_experiment(self, true_probs):
        '''Run one experiment; return (method, successes, failures) to share.'''
        m       = self.best_method()
        success = int(np.random.random() < true_probs[m])
        if success:
            self.hits[m]   += 1
        else:
            self.misses[m] += 1
        return m, success, 1 - success

    def receive_evidence(self, method, successes, failures):
        '''
        EXERCISE 1: update beliefs with a neighbor's experimental results.

        Bayesian update: add the neighbor's observed successes and failures to our
        running totals. This is exact because the Beta distribution is the conjugate
        prior for Bernoulli data — all evidence combines by adding counts.
        '''
        # YOUR CODE HERE
        # ---------------------------------------------------------------
        # HINT (uncomment to reveal the solution):
        # self.hits[method]   += successes
        # self.misses[method] += failures
        # ---------------------------------------------------------------
        pass

> **Check your work** — After writing your solution, run the demo cell below.
>
> **Expected output**: after the neighbor reports 8 successes and 1 failure on method B,
> B's belief should jump from ~0.50 to ~0.75 and the preferred method should flip to B.
>
> If you're stuck, uncomment the `# HINT` lines in the exercise cell above.

In [ ]:
# Quick demo: watch beliefs shift after one neighbor's report
s = Scientist()
print('Starting beliefs:  A={:.2f}, B={:.2f}'.format(s.beliefs[0], s.beliefs[1]))
print('Preferred method:', 'A' if s.best_method() == 0 else 'B')

# Neighbor reports: B succeeded 8 times, failed once
s.receive_evidence(method=1, successes=8, failures=1)

print()
print('After neighbor reports 8 successes, 1 failure on method B:')
print('Updated beliefs:   A={:.2f}, B={:.2f}'.format(s.beliefs[0], s.beliefs[1]))
print('Preferred method:', 'A' if s.best_method() == 0 else 'B')
print()
print('One report flipped the preference. Now imagine this propagating across a dense network.')

### Communication Network Topologies

We test three topologies that represent different levels of connectivity:

| Topology | Description | Speed of information spread |
|----------|-------------|-----------------------------|
| **Complete** | Every scientist connected to every other | Instant — everyone sees all results |
| **Cycle** | Scientists form a ring; each talks to 2 neighbors | Slow — results spread locally |
| **Wheel** | One hub connected to all, plus a ring | Fast through hub, slow on periphery |

Zollman's key insight: the **complete** network — the most transparent — can
*harm* collective discovery. Faster information spread means everyone converges
on early noisy evidence before the true signal has time to accumulate.

In [ ]:
def make_network(topology, n):
    if topology == 'complete': return nx.complete_graph(n)
    if topology == 'cycle':    return nx.cycle_graph(n)
    if topology == 'wheel':    return nx.wheel_graph(n)
    raise ValueError('Unknown topology: ' + topology)


fig, axes = plt.subplots(1, 3, figsize=(13, 4))
descs = {
    'complete': 'Complete\n(everyone talks to everyone)',
    'cycle':    'Cycle\n(ring of neighbors)',
    'wheel':    'Wheel\n(hub + ring)',
}
for ax, topo in zip(axes, ['complete', 'cycle', 'wheel']):
    G   = make_network(topo, 10)
    pos = nx.circular_layout(G)
    nx.draw(G, pos, ax=ax, node_size=350, node_color='steelblue',
            edge_color='#aaaaaa', with_labels=False)
    ax.set_title(descs[topo], fontsize=10)

plt.suptitle('Three Communication Network Topologies', fontsize=13)
plt.tight_layout()
plt.show()

### How the Simulation Works

Each round has two steps:

**Step 1 — Experiment**
Every scientist uses the method they currently believe is better.
They observe one outcome: success (1) or failure (0).

**Step 2 — Share + Update**
Each scientist shares their result with their **network neighbors only**
(not the whole community). Neighbors incorporate that result using `receive_evidence`.

After many rounds the community either:
- **Converges correctly** — everyone on method B (the right answer) ✓
- **Converges incorrectly** — everyone on method A (locked in on wrong answer) ✗
- **No convergence** — community remains split

The Zollman Effect appears in the ratio of correct to incorrect convergences
across different topologies.

In [ ]:
def run_simulation(n, topology, p_A, p_B, n_rounds):
    '''Run one simulation. Returns (fraction-on-B history, converged_correctly).'''
    G          = make_network(topology, n)
    scientists = [Scientist() for _ in range(n)]
    true_probs = [p_A, p_B]
    history    = []

    for _ in range(n_rounds):
        # Everyone runs one experiment
        results = [s.run_experiment(true_probs) for s in scientists]

        # Share with neighbors only (not everyone!)
        for i, s in enumerate(scientists):
            for j in G.neighbors(i):
                method, succ, fail = results[j]
                s.receive_evidence(method, succ, fail)

        n_on_B = sum(1 for s in scientists if s.best_method() == 1)
        history.append(n_on_B / n)

    final = [s.best_method() for s in scientists]
    if   all(m == 1 for m in final): converged = True   # correct
    elif all(m == 0 for m in final): converged = False  # wrong
    else:                            converged = None   # no consensus

    return history, converged


def run_many(n, topology, p_A, p_B, n_sims, n_rounds):
    '''Run n_sims simulations; return aggregate statistics.'''
    n_correct = n_wrong = 0
    histories = []
    for _ in range(n_sims):
        hist, conv = run_simulation(n, topology, p_A, p_B, n_rounds)
        histories.append(hist)
        if conv is True:  n_correct += 1
        if conv is False: n_wrong   += 1
    return {
        'p_correct': n_correct / n_sims,
        'p_wrong':   n_wrong   / n_sims,
        'p_none':    (n_sims - n_correct - n_wrong) / n_sims,
        'mean_traj': np.mean(histories, axis=0),
    }

In [ ]:
# Run main simulations for all three topologies.
# p_B is only 0.001 better than p_A — tiny signal, hard to detect by chance.
P_A, P_B = 0.500, 0.501

print('Running Part 1 simulations...')
z_results = {}
for topo in ['complete', 'cycle', 'wheel']:
    print('  {}...'.format(topo), end=' ', flush=True)
    z_results[topo] = run_many(N_SCIENTISTS, topo, P_A, P_B, N_SIMS, N_ROUNDS_Z)
    print('correct={:.2f}  wrong={:.2f}'.format(
        z_results[topo]['p_correct'], z_results[topo]['p_wrong']))

print('Done.')

In [ ]:
# Plot 1: Convergence outcomes per topology
topos  = ['complete', 'cycle', 'wheel']
labels = ['Complete', 'Cycle', 'Wheel']
x = np.arange(len(topos))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, [z_results[t]['p_correct'] for t in topos], w,
       label='Correct (converged to B)', color='steelblue')
ax.bar(x + w/2, [z_results[t]['p_wrong']   for t in topos], w,
       label='Wrong (converged to A)',   color='tomato')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('Fraction of simulations', fontsize=11)
ax.set_ylim(0, 1)
ax.set_title(
    'Which network finds the better method?  '
    '({} scientists, {} sims, p_B-p_A=0.001)'.format(N_SCIENTISTS, N_SIMS),
    fontsize=11
)
ax.legend(fontsize=10)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Mean trajectory over time
colors = {'complete': 'tomato', 'cycle': 'steelblue', 'wheel': 'seagreen'}

fig, ax = plt.subplots(figsize=(10, 5))
for topo in topos:
    ax.plot(z_results[topo]['mean_traj'],
            color=colors[topo], label=topo.capitalize(), linewidth=2)

ax.axhline(1.0, color='black', linestyle='--', alpha=0.15, linewidth=1)
ax.axhline(0.0, color='black', linestyle=':',  alpha=0.15, linewidth=1)
ax.set_xlabel('Round', fontsize=11)
ax.set_ylabel('Average fraction using method B', fontsize=11)
ax.set_title('Average trajectory across all simulations', fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(-0.02, 1.05)
plt.tight_layout()
plt.show()

### Interpreting the Zollman Effect

**Complete network**: everyone sees the same evidence. If method A happened to look
better in the first few rounds (random chance), that signal propagates instantly
to everyone. The community locks in on A before the true signal for B accumulates.
With p\_B - p\_A = 0.001, the signal-to-noise ratio is very low.

**Cycle network**: scientists only see 2 neighbors' results. Different clusters
maintain independent estimates longer. Some groups keep exploring B even when
nearby groups have converged on A. This diversity gives B more chances to prove itself.

**Wheel network**: the hub accelerates convergence; peripheral scientists are
partially insulated. Performance falls between complete and cycle.

### When Does Topology Stop Mattering?

The Zollman gap should shrink as method B becomes increasingly better.
When the signal is strong, even the complete network finds B reliably.
The next cell sweeps across effect sizes to show this transition.

In [ ]:
# Effect size sweep: how does P(correct) vary with p_B - p_A?
print('Running effect size sweep...')
sweep = {'complete': [], 'cycle': []}

for delta in DELTA_SWEEP:
    for topo in ['complete', 'cycle']:
        r = run_many(N_SCIENTISTS, topo, 0.5, 0.5 + delta,
                     n_sims=max(50, N_SIMS // 2), n_rounds=N_ROUNDS_Z)
        sweep[topo].append(r['p_correct'])
    print('  delta={} done'.format(delta))

fig, ax = plt.subplots(figsize=(9, 5))
xs = range(len(DELTA_SWEEP))
ax.plot(xs, sweep['complete'], color='tomato',    marker='o', linewidth=2, label='Complete')
ax.plot(xs, sweep['cycle'],    color='steelblue', marker='o', linewidth=2, label='Cycle')

ax.set_xticks(xs)
ax.set_xticklabels([str(d) for d in DELTA_SWEEP], fontsize=9)
ax.set_xlabel('p_B - p_A  (how much better method B is)', fontsize=11)
ax.set_ylabel('P(correct convergence)', fontsize=11)
ax.set_title('The Zollman gap disappears when evidence is strong', fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print()
print('Cycle advantage is largest at small delta (ambiguous evidence).')
print('At large delta both networks reliably find the right answer.')

### What Part 1 Established

- Even perfectly rational Bayesian agents can produce collective epistemic failure
- The *structure* of communication determines collective outcomes, not just content
- Dense communication = fast lock-in on noisy early evidence

**Zollman's policy implication**: "Transient diversity" — communities should share
results *less* early on, more once the signal is clearer. This maps to real
institutions: preregistration, blind review, and staged replication all slow
information flow in the early discovery phase.

---

## Part 2: Better than Best — NK Landscapes

Part 1 was about *communication structure*. Part 2 is about *research strategy*:
not how scientists share results, but how they decide which approach to try next.

**The puzzle (Wu 2019)**: if one lab's approach is clearly the best, shouldn't
everyone copy them? Intuitively yes — but on hard problems, this converges too
fast and gets stuck on local optima. A community that includes some scientists
following *any* improvement (not just the best) can outperform one that all
copies the leader.

### The NK Landscape Model

A **fitness landscape** maps every possible theory to a quality score.

**Theory**: a binary string of length N. Each bit represents a scientific
assumption (0 = false, 1 = true).
Example: `[1, 0, 1, 1, 0, 0, 1, 0]` — one research paradigm.

**Fitness**: a number between 0 and 1. Solving the scientific problem = finding
the theory with the highest fitness.

**Ruggedness (K)**: each bit's fitness contribution depends on K other bits.
- K = 0: smooth — one peak, hill-climbing always finds it
- K = N-1: maximally rugged — many local peaks, hill-climbing gets stuck

Real scientific problems lie somewhere in the middle. NK lets us vary ruggedness
and measure which strategy works best at each level.

**Two update strategies** (from Wu 2019):
- **Best**: copy the single highest-fitness peer if they beat you
- **Better**: copy *any* peer who beats you, chosen at random
- **Mixed**: a community with some Best and some Better agents

In [ ]:
class NKLandscape:
    '''
    Fitness landscape over binary strings of length N.
    Each bit i has a fitness contribution that depends on K other bits.
    K=0: smooth (one peak). K near N-1: maximally rugged (many local peaks).
    '''

    def __init__(self, N, K, seed=None):
        assert 0 <= K < N, 'K must be between 0 and N-1'
        self.N, self.K = N, K
        rng = np.random.RandomState(seed)

        # For each bit i, choose which K other bits it depends on
        self.interactions = []
        for i in range(N):
            others = [j for j in range(N) if j != i]
            deps   = sorted(rng.choice(others, K, replace=False).tolist())
            self.interactions.append([i] + deps)  # [self, dep1, dep2, ...]

        # Random fitness table: 2^(K+1) entries per bit
        self.tables = [rng.random(2 ** (K + 1)) for _ in range(N)]

    def fitness(self, theory):
        '''Fitness of a theory (list of N bits). Average of per-bit contributions.'''
        total = 0.0
        for i in range(self.N):
            bits  = tuple(theory[j] for j in self.interactions[i])
            idx   = int(''.join(map(str, bits)), 2)
            total += self.tables[i][idx]
        return total / self.N

    def neighbors(self, theory):
        '''All theories reachable by flipping exactly one bit.'''
        result = []
        for i in range(self.N):
            nb    = list(theory)
            nb[i] = 1 - nb[i]
            result.append(nb)
        return result

    def is_local_max(self, theory):
        '''True if no single-bit flip improves fitness.'''
        f = self.fitness(theory)
        return all(self.fitness(nb) <= f for nb in self.neighbors(theory))

    def count_local_maxima(self):
        '''Count all local maxima by exhaustive search (feasible for N <= 16).'''
        return sum(1 for bits in product([0, 1], repeat=self.N)
                   if self.is_local_max(list(bits)))

    def global_max(self):
        '''Find the global maximum (exhaustive; feasible for N <= 18).'''
        best_f, best_t = -1.0, None
        for bits in product([0, 1], repeat=self.N):
            f = self.fitness(list(bits))
            if f > best_f:
                best_f, best_t = f, list(bits)
        return best_t, best_f

In [ ]:
# Quick demo: explore a small NK landscape
np.random.seed(42)
L = NKLandscape(N=8, K=3, seed=42)

theory = [1, 0, 1, 1, 0, 0, 1, 0]
print('Theory:   ', theory)
print('Fitness:   {:.3f}'.format(L.fitness(theory)))

nb_fits = [L.fitness(nb) for nb in L.neighbors(theory)]
_, gmax = L.global_max()

print('Best neighbor fitness: {:.3f}'.format(max(nb_fits)))
print('Global max fitness:    {:.3f}'.format(gmax))
print('This theory is at {:.0%} of the global max.'.format(L.fitness(theory) / gmax))
print()
if max(nb_fits) > L.fitness(theory):
    print('Not a local maximum: one bit-flip can improve fitness.')
else:
    print('Local maximum: no single bit-flip can help. Hill-climbing is stuck.')

In [ ]:
# How does the number of local optima grow with K?
K_vals  = [k for k in K_SWEEP if k < N_BITS]
avg_lm  = []

print('Counting local optima for each K...')
for K in K_vals:
    counts = [NKLandscape(N=N_BITS, K=K, seed=s).count_local_maxima()
              for s in range(N_RUGGED)]
    avg_lm.append(np.mean(counts))
    print('  K={}: avg {:.1f} local optima'.format(K, avg_lm[-1]))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(K_vals, avg_lm, color='steelblue', edgecolor='white')
ax.set_xlabel('K  (epistatic interactions per bit)', fontsize=11)
ax.set_ylabel('Average local optima', fontsize=11)
ax.set_title(
    'Ruggedness grows with K  (N={}, {} landscapes)'.format(N_BITS, N_RUGGED),
    fontsize=11
)
ax.set_xticks(K_vals)
plt.tight_layout()
plt.show()

print()
print('K=0: one local optimum = global max. Hill-climbing always succeeds.')
print('High K: many local optima. Hill-climbing gets trapped almost every run.')

### Two Research Strategies

**The 'Best' strategy — copy the leader:**
If any peer has higher fitness than me, copy the single best one.
Fast convergence. Good on smooth landscapes. Gets trapped on local optima.

**The 'Better' strategy — copy any improvement:**
From all peers who outperform me, pick one *at random* and copy them.
Slower convergence. The randomness is the point.

**Why does randomness help?**
When everyone copies the same leader, the entire community converges to one
position on the landscape. If that position is a local optimum, no one can escape.
When agents copy *different* better peers at random, the community maintains spread
across multiple positions — some of which are higher ground. The community
collectively samples the landscape while still moving upward.

**Exercise 2**: implement `update_better` in `ScientistNK`.

In [ ]:
class ScientistNK:
    '''A scientist exploring an NK landscape through local search and social learning.'''

    def __init__(self, landscape):
        self.landscape = landscape
        self.theory    = list(np.random.randint(0, 2, landscape.N))  # random start
        self.fit       = landscape.fitness(self.theory)

    def explore_locally(self):
        '''Flip one random bit; keep it only if it improves fitness (hill-climbing).'''
        i        = np.random.randint(0, self.landscape.N)
        trial    = list(self.theory)
        trial[i] = 1 - trial[i]
        f_trial  = self.landscape.fitness(trial)
        if f_trial > self.fit:
            self.theory = trial
            self.fit    = f_trial

    def update_best(self, peer_theories, peer_fitnesses):
        '''Copy the single best peer if they outperform me.'''
        if not peer_fitnesses: return
        best = int(np.argmax(peer_fitnesses))
        if peer_fitnesses[best] > self.fit:
            self.theory = list(peer_theories[best])
            self.fit    = peer_fitnesses[best]

    def update_better(self, peer_theories, peer_fitnesses):
        '''
        EXERCISE 2: copy a random peer who outperforms me.

        Find all peers doing better than me, then choose one uniformly at random.
        This is the key difference from update_best: we do NOT always follow the
        same leader. Different scientists end up following different better peers,
        so the community maintains spread across the landscape.
        '''
        # YOUR CODE HERE
        # ---------------------------------------------------------------
        # HINT (uncomment to reveal the solution):
        # better = [i for i, f in enumerate(peer_fitnesses) if f > self.fit]
        # if better:
        #     chosen      = np.random.choice(better)
        #     self.theory = list(peer_theories[chosen])
        #     self.fit    = peer_fitnesses[chosen]
        # ---------------------------------------------------------------
        pass

> **Check your work** — After writing your solution, run the verification cell below.
>
> If you're stuck, uncomment the `# HINT` lines in the exercise cell above.

In [ ]:
# Quick check for Exercise 2: verifies update_better copies a better peer
_L = NKLandscape(N=4, K=1, seed=99)
_agent = ScientistNK(_L)
_agent.theory = [0, 0, 0, 0]
_agent.fit    = _L.fitness([0, 0, 0, 0])
_base_fit = _agent.fit

# One worse peer + two better peers
_peer_theories  = [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 1]]
_peer_fitnesses = [_base_fit - 0.1, _base_fit + 0.05, _base_fit + 0.10]

np.random.seed(0)
_agent.update_better(_peer_theories, _peer_fitnesses)

assert _agent.fit > _base_fit, 'update_better should copy a better peer'
assert _agent.theory in [[0, 1, 0, 0], [0, 0, 1, 1]], 'should only choose from better peers'
print('Exercise 2 check passed!')
print('  Copied theory:', _agent.theory, '  Fitness: {:.3f}'.format(_agent.fit))
print('  (Either of the two better peers is correct.)')

In [ ]:
def run_community(landscape, n_agents, n_rounds, strategy, mix_ratio=0.5):
    '''
    Simulate a community of scientists on an NK landscape.

    Each round:
      1. Every scientist tries a local improvement (explore_locally)
      2. Scientists share their current theories and fitnesses
      3. Each scientist decides whether to copy a peer (social update)

    strategy:  'best', 'better', or 'mixed'
    mix_ratio: fraction using 'better' when strategy='mixed'
    Returns:   (history of mean fitness per round, final mean fitness)
    '''
    agents = [ScientistNK(landscape) for _ in range(n_agents)]

    if strategy == 'mixed':
        strats = ['better' if np.random.random() < mix_ratio else 'best'
                  for _ in range(n_agents)]
    else:
        strats = [strategy] * n_agents

    history = [np.mean([a.fit for a in agents])]

    for _ in range(n_rounds):
        # Step 1: local exploration
        for a in agents:
            a.explore_locally()

        # Step 2: social update
        theories  = [a.theory for a in agents]
        fitnesses = [a.fit    for a in agents]

        for i, (agent, strat) in enumerate(zip(agents, strats)):
            pts = theories[:i]  + theories[i+1:]   # exclude self
            pfs = fitnesses[:i] + fitnesses[i+1:]
            if strat == 'best':
                agent.update_best(pts, pfs)
            else:
                agent.update_better(pts, pfs)

        history.append(np.mean([a.fit for a in agents]))

    return history, history[-1]

In [ ]:
# Compare the three strategies on smooth (K=1) and rugged (K=6) landscapes.
# Average over N_TRIALS independent landscapes to get stable results.
K_RUGGED = min(6, N_BITS - 2)   # cap at N-2 so the landscape is valid

print('Running strategy comparison...')
smooth_trajs = {'best': [], 'better': [], 'mixed': []}
rugged_trajs = {'best': [], 'better': [], 'mixed': []}

for seed in range(N_TRIALS):
    # Smooth landscape
    ls = NKLandscape(N=N_BITS, K=1, seed=seed)
    _, max_s = ls.global_max()
    for strat in ['best', 'better', 'mixed']:
        hist, _ = run_community(ls, N_AGENTS, N_ROUNDS_NK, strat)
        smooth_trajs[strat].append([x / max_s for x in hist])

    # Rugged landscape
    lr = NKLandscape(N=N_BITS, K=K_RUGGED, seed=seed)
    _, max_r = lr.global_max()
    for strat in ['best', 'better', 'mixed']:
        hist, _ = run_community(lr, N_AGENTS, N_ROUNDS_NK, strat)
        rugged_trajs[strat].append([x / max_r for x in hist])

    if (seed + 1) % 5 == 0:
        print('  {}/{} landscapes done'.format(seed + 1, N_TRIALS))

print('Done.')

In [ ]:
# Plot: smooth vs rugged landscape strategies
sc = {'best': 'tomato', 'better': 'steelblue', 'mixed': 'seagreen'}
sl = {'best': 'Best (copy leader)', 'better': 'Better (copy any improvement)', 'mixed': 'Mixed (50/50)'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for strat in ['best', 'better', 'mixed']:
    ms = np.mean(smooth_trajs[strat], axis=0)
    mr = np.mean(rugged_trajs[strat], axis=0)
    ax1.plot(ms, color=sc[strat], label=sl[strat], linewidth=2)
    ax2.plot(mr, color=sc[strat], label=sl[strat], linewidth=2)

ax1.set_title('Smooth landscape (K=1) — all strategies similar', fontsize=11)
ax2.set_title('Rugged landscape (K={}) — mixed community wins'.format(K_RUGGED), fontsize=11)

for ax in [ax1, ax2]:
    ax.set_xlabel('Round', fontsize=11)
    ax.set_ylabel('Mean fitness / global max', fontsize=10)
    ax.legend(fontsize=9)
    ax.set_ylim(0.3, 1.05)
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.3)

plt.suptitle('N={} bits, {} agents, {} landscapes'.format(N_BITS, N_AGENTS, N_TRIALS), fontsize=11)
plt.tight_layout()
plt.show()

# Final fitness summary table
print('Final fitness (fraction of global max):')
print('{:<10}  {:>14}  {:>14}'.format('Strategy', 'K=1 (smooth)', 'K={} (rugged)'.format(K_RUGGED)))
print('-' * 42)
for strat in ['best', 'better', 'mixed']:
    s = np.mean([t[-1] for t in smooth_trajs[strat]])
    r = np.mean([t[-1] for t in rugged_trajs[strat]])
    print('{:<10}  {:>14.3f}  {:>14.3f}'.format(strat, s, r))

In [ ]:
# Full K sweep: where exactly does the mixed advantage appear and peak?
print('Running K sweep...')
sweep_finals = {'best': [], 'better': [], 'mixed': []}

for K in K_vals:   # K_vals was defined in the ruggedness cell
    k_finals = {'best': [], 'better': [], 'mixed': []}
    for seed in range(N_TRIALS):
        land = NKLandscape(N=N_BITS, K=K, seed=seed)
        _, mf = land.global_max()
        for strat in ['best', 'better', 'mixed']:
            _, final = run_community(land, N_AGENTS, N_ROUNDS_NK, strat)
            k_finals[strat].append(final / mf)
    for strat in ['best', 'better', 'mixed']:
        sweep_finals[strat].append(np.mean(k_finals[strat]))
    print('  K={}: best={:.3f}  better={:.3f}  mixed={:.3f}'.format(
        K, sweep_finals['best'][-1], sweep_finals['better'][-1], sweep_finals['mixed'][-1]))

print('Done.')

In [ ]:
# Plot the K sweep
fig, ax = plt.subplots(figsize=(10, 5))

for strat in ['best', 'better', 'mixed']:
    ax.plot(K_vals, sweep_finals[strat],
            color=sc[strat], label=sl[strat],
            marker='o', linewidth=2, markersize=7)

ax.set_xlabel('K  (landscape ruggedness)', fontsize=11)
ax.set_ylabel('Final fitness / global max', fontsize=11)
ax.set_title(
    'Mixed advantage across all ruggedness levels  (N={}, {} agents, {} landscapes)'.format(
        N_BITS, N_AGENTS, N_TRIALS),
    fontsize=11
)
ax.legend(fontsize=10)
ax.set_xticks(K_vals)
ax.set_ylim(0.4, 1.05)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print()
print('K=0: all strategies equal — one peak, hill-climbing always finds it.')
print('K=3 to K={}: mixed community consistently outperforms pure strategies.'.format(K_RUGGED))
print('Very high K: all strategies struggle — landscape too rugged for any approach.')

---

## Synthesis: The Two Parts Together

Both models show different faces of the same underlying claim:
**individual rationality does not guarantee collective rationality**.

**Part 1 (Zollman)** — the level of *communication*:
Each scientist is a perfect Bayesian, yet the network structure determines whether
the community reaches the correct consensus. Dense networks amplify early noise
into permanent false beliefs.

**Part 2 (Wu)** — the level of *strategy*:
Following the single best peer is individually sensible, but the community locks
in on local optima. A community that *deliberately* includes less-greedy agents
outperforms a homogeneous one — on hard problems.

Together: **epistemic diversity is a structural feature of good collective
reasoning, not just a political preference**.

---

## Discussion Questions

1. **Applying Zollman**: A tech company considers publishing all internal A/B
   test results to a shared real-time dashboard. Using the Zollman model,
   predict when this helps vs. harms collective discovery.

2. **Better than Best**: If the 'Better' strategy works well, why doesn't
   everyone adopt it? What publication and funding incentives push toward 'Best'?

3. **Connecting the two**: Both models assume rational agents. What happens when
   agents are systematically biased — e.g., by prestige, grant pressure, or
   publication bias? Does diversity help more or less under systematic bias?

4. **AI and scientific literature**: Language models are trained on published
   science. If a field converged prematurely (Zollman Effect) on some topic,
   what does the model learn? Does AI summarization make this better or worse?

5. **Mixing ratio**: The simulation used 50/50 Best/Better in the mixed community.
   Is 50/50 always optimal? Change `mix_ratio` in `run_community` and report
   what ratio maximizes community fitness on a K=5 landscape.

---

**Author:** [Aniket Ghosh](https://www.linkedin.com/in/aniketghosh-/)